## Review Lab 2 (Coordinate Transformation)

In [1]:
import pandas as pd  
import pyproj
import pygeodesy
from pygeodesy.ellipsoidalKarney import LatLon
from pygeodesy import Ellipsoids
import math

#### ข้อ 1 แปลง ECEF -> Geodetic, UTM (47,48) และหา Orthometric Height (from TGM2017)

In [2]:
def xyz2lla(x,y,z):
    ecef = '+proj=geocent +ellps=WGS84'
    lla = "EPSG:4326"
    transproj = pyproj.Transformer.from_crs(ecef, lla)
    lat,lon,ell_h = transproj.transform(x,y,z, radians = False)
    return lat, lon, ell_h

def lla2utm(lat,lon,ell_h,zone):
    lla = "epsg:4326"
    utm = "epsg:326" + str(zone)
    transproj = pyproj.Transformer.from_crs(lla, utm, always_xy=True)
    E, N, h = transproj.transform(lon, lat, ell_h, radians = False)
    return E, N, h

def Geoid_seperation(lat, lon):
    geoid_model = r"C:\Lab_Geodetic_Survey\Midterm_Prepare\tgm2017-1_2.pgm"
    geoid_interpolator = pygeodesy.GeoidKarney(geoid_model)
    pos = LatLon(lat, lon)
    N = geoid_interpolator(pos)
    return N

In [3]:
ASKN_lla = xyz2lla(-1482251.762,5925272.273,1831475.943)
AMKO_lla = xyz2lla(-883143.728,6010905.970,1937627.074)
APKN_lla = xyz2lla(-1037944.776,6165673.259,1255828.562)
print(ASKN_lla)
print(AMKO_lla)
print(APKN_lla)

(16.797833249592017, 104.0447406266078, 172.46500600967556)
(17.80063864793367, 98.35830022803451, 783.8907293761149)
(11.431535053995404, 99.55571916097857, -3.2311080200597644)


In [4]:
ASKN_UTM_47 = lla2utm(ASKN_lla[0],ASKN_lla[1],ASKN_lla[2],47)
AMKO_UTM_47 = lla2utm(AMKO_lla[0],AMKO_lla[1],AMKO_lla[2],47)
APKN_UTM_47 = lla2utm(APKN_lla[0],APKN_lla[1],APKN_lla[2],47)
print(ASKN_UTM_47)
print(AMKO_UTM_47)
print(APKN_UTM_47)

(1038134.1514753288, 1864049.7897068574, 172.46500600967556)
(431990.8901367701, 1968245.6317819764, 783.8907293761149)
(560619.7615189415, 1263753.417720001, -3.2311080200597644)


In [5]:
ASKN_UTM_48 = lla2utm(ASKN_lla[0],ASKN_lla[1],ASKN_lla[2],48)
AMKO_UTM_48 = lla2utm(AMKO_lla[0],AMKO_lla[1],AMKO_lla[2],48)
APKN_UTM_48 = lla2utm(APKN_lla[0],APKN_lla[1],APKN_lla[2],48)
print(ASKN_UTM_48)
print(AMKO_UTM_48)
print(APKN_UTM_48)

(398206.75845628756, 1857435.8112767697, 172.46500600967556)
(-205186.8772905207, 1980664.229656586, 783.8907293761149)
(-94702.92075593548, 1269307.6391286352, -3.2311080200597644)


In [6]:
ASKN_undulation = Geoid_seperation(ASKN_lla[0], ASKN_lla[1])
AMKO_undulation = Geoid_seperation(AMKO_lla[0],AMKO_lla[1])
APKN_undulation= Geoid_seperation(APKN_lla[0],APKN_lla[1])
#print(ASKN_undulation)
#print(AMKO_undulation)
#print(APKN_undulation)
ASKN_Ortho = ASKN_lla[2] - ASKN_undulation
AMKO_Ortho = AMKO_lla[2] - AMKO_undulation
APKN_Ortho = APKN_lla[2] - APKN_undulation
print(ASKN_Ortho)
print(AMKO_Ortho)
print(APKN_Ortho)

197.86216371202693
822.0888481201179
25.478643636187225


#### ข้อ 2 Create function for Coordinate Transformation (UTM2LLA, UTM2XYZ, LLA2XYZ, XYZ2UTM)

In [7]:
#UTM2LLA
def UTM2LLA(E, N, h, zone):
    utm = "epsg:326" + str(zone)
    lla = "EPSG:4326"
    transproj = pyproj.Transformer.from_crs(utm, lla, always_xy=True)

    lon,lat,ell_h = transproj.transform(E,N,h, radians = False)

    return lon, lat, ell_h

In [8]:
#UTM2XYZ
def UTM2XYZ(E, N, h, zone):
    utm = "epsg:326" + str(zone)
    ecef = '+proj=geocent +ellps=WGS84'
    transproj = pyproj.Transformer.from_crs(utm, ecef, always_xy=True)

    x,z,y = transproj.transform(E,N,h)

    return x, y, z

In [9]:
#LLA2XYZ
def LLA2XYZ(lat, lon, ell_h):
    lla = "epsg:4326"
    ecef = '+proj=geocent +ellps=WGS84'
    transproj = pyproj.Transformer.from_crs(lla, ecef, always_xy=True)

    x,y,z = transproj.transform(lon, lat, ell_h, radians = False)

    return x, y, z

In [10]:
#XYZ2UTM
def XYZ2UTM(x, y, z, zone):
    ecef = '+proj=geocent +ellps=WGS84'
    utm = "epsg:326" + str(zone)
    transproj = pyproj.Transformer.from_crs(ecef, utm, always_xy=True)

    E,N,h = transproj.transform(x,y,z)

    return E, N, h

#### ข้อ 3 Coordinate Transformation (ECEF -> Geodetic and UTM)

In [11]:
df = pd.read_csv("coordinate.csv")

In [12]:
df.head()

,sta,x,y,z
0,AMKO,-8.831437e+05,6010905.970,1.937627e+06
1,AUPG,-9.448123e+05,6059317.812,1.748566e+06
2,AWLK,-9.568981e+05,6237552.826,9.230742e+05
3,AYYA,-1.134356e+06,6075002.177,1.572057e+06
4,BDNG,-1.394062e+06,5916171.632,1.926911e+06


In [13]:
lat, lon, ell_h = xyz2lla(df['x'],df['y'],df['z'])
df["lat"] = lat
df["lon"] = lon
df["ell_h"] = ell_h

In [14]:
df

,sta,x,y,z,lat,lon,ell_h
0,AMKO,-8.831437e+05,6010905.970,1.937627e+06,17.800639,98.358300,783.890771
1,AUPG,-9.448123e+05,6059317.812,1.748566e+06,16.016167,98.862603,427.829051
2,AWLK,-9.568981e+05,6237552.826,9.230742e+05,8.377234,98.721705,-8.143341
3,AYYA,-1.134356e+06,6075002.177,1.572057e+06,14.364311,100.576764,-15.464410
4,BDNG,-1.394062e+06,5916171.632,1.926911e+06,17.700826,103.259067,146.757909
...,...,...,...,...,...,...,...
129,WGCN,-1.012833e+06,5986316.617,1.947860e+06,17.899837,99.603011,71.769541
130,WHAG,-9.029830e+05,5944737.504,2.122260e+06,19.561554,98.636989,704.714909
131,WNKH,-1.268879e+06,6047265.500,1.577808e+06,14.417009,101.850282,399.657755
132,WNNW,-1.445454e+06,5906210.364,1.919621e+06,17.631717,103.751956,141.625062


In [15]:
E, N, h = lla2utm(df["lat"],df["lon"],df["ell_h"],47)
df["E_utm47"] = E
df["N_utm47"] = N
df["h_utm47"] = h

In [16]:
df

,sta,x,y,z,lat,lon,ell_h,E_utm47,N_utm47,h_utm47
0,AMKO,-8.831437e+05,6010905.970,1.937627e+06,17.800639,98.358300,783.890771,4.319909e+05,1.968246e+06,783.890771
1,AUPG,-9.448123e+05,6059317.812,1.748566e+06,16.016167,98.862603,427.829051,4.853008e+05,1.770729e+06,427.829051
2,AWLK,-9.568981e+05,6237552.826,9.230742e+05,8.377234,98.721705,-8.143341,4.693609e+05,9.260129e+05,-8.143341
3,AYYA,-1.134356e+06,6075002.177,1.572057e+06,14.364311,100.576764,-15.464410,6.700232e+05,1.588598e+06,-15.464410
4,BDNG,-1.394062e+06,5916171.632,1.926911e+06,17.700826,103.259067,146.757909,9.519718e+05,1.962201e+06,146.757909
...,...,...,...,...,...,...,...,...,...,...
129,WGCN,-1.012833e+06,5986316.617,1.947860e+06,17.899837,99.603011,71.769541,5.638732e+05,1.979207e+06,71.769541
130,WHAG,-9.029830e+05,5944737.504,2.122260e+06,19.561554,98.636989,704.714909,4.619230e+05,2.163004e+06,704.714909
131,WNKH,-1.268879e+06,6047265.500,1.577808e+06,14.417009,101.850282,399.657755,8.073528e+05,1.595751e+06,399.657755
132,WNNW,-1.445454e+06,5906210.364,1.919621e+06,17.631717,103.751956,141.625062,1.004564e+06,1.955789e+06,141.625062


#### ข้อ 4 Design TM for Chula

In [17]:
# (13.739536893607612, 100.52973308478612)

def get_Rm_for_LDP(lat):
    # Mean Radius of Curvature
    e = Ellipsoids.WGS84
    return e.rocGauss(lat)

r = get_Rm_for_LDP(13.739536893607612)
r

6359153.733677555

In [18]:
# ต้องรู้ h (ellipsoidal height)
# esf = r / (r + h)
# k0 = 1 / esf = (r + h) / r

In [19]:
def TM(lat,lon,ell_h):
    tm = 'proj=tmerc +lat_0=0 +lon_0=100.52973308478612 +k = 1 / esf +x_0=500000 +y_0=0 +ellps=WGS84 +units=m +no_defs'
    lla = "epsg:4326"
    transproj = pyproj.Transformer.from_crs(lla, tm, always_xy=True)
    E,N,h = transproj.transform(lon, lat, ell_h, radians = False)
    return E,N,h

#### ข้อ 5 Find Distance between A - B 

In [20]:
# ECEF (จุด A) -> -1482251.762, 5926272.273, 1831475.943
# ECEF (จุด B) -> -1037944.776, 6165673.259, 1255828.562

In [21]:
A_lla = xyz2lla(-1482251.762, 5926272.273, 1831475.943)
B_lla = xyz2lla(-1037944.776, 6165673.259, 1255828.562)

In [22]:
# หา ESF
h_A = A_lla[2]
h_B = B_lla[2]

h_avg = (h_A + h_B) / 2

ESF = 6371.1 / (6371.1 + (h_avg/1000))

In [23]:
#หา GSF (K') จุดเริ่มต้น
def utm_scale_factor(lat, lon, zone):
    prj = pyproj.Proj('epsg:326' + str(zone))
    fc = prj.get_factors(lon, lat, radians=False)
    scaleX = fc.parallel_scale
    scaleY = fc.meridional_scale
    return scaleX, scaleY

scaleX_A_47, scaleY_A_47 = utm_scale_factor(A_lla[0],A_lla[1],47)
scaleX_A_48, scaleY_A_48 = utm_scale_factor(A_lla[0],A_lla[1],48)

scaleX_B_47, scaleY_B_47 = utm_scale_factor(B_lla[0],B_lla[1],47)
scaleX_B_48, scaleY_B_48 = utm_scale_factor(B_lla[0],B_lla[1],48)


##### Distance on Grid UTM ***

In [24]:
A_utm47 = lla2utm(A_lla[0], A_lla[1], A_lla[2], '47')
B_utm47 = lla2utm(B_lla[0], B_lla[1], B_lla[2], '47')
A_utm48 = lla2utm(A_lla[0], A_lla[1], A_lla[2], '48')
B_utm48 = lla2utm(B_lla[0], B_lla[1], B_lla[2], '48')

grid_dist_utm47 = math.sqrt((A_utm47[0] - B_utm47[0])**2 + (A_utm47[1] - B_utm47[1])**2)
grid_dist_utm48 = math.sqrt((A_utm48[0] - B_utm48[0])**2 + (A_utm48[1] - B_utm48[1])**2)

print(grid_dist_utm47)
print(grid_dist_utm48)

766684.6661817186
766998.1291879389


In [25]:
# หาจุดตรงกลางก่อน จะได้หา K' ต่อได้
E_mid_47 = (A_utm47[0] + B_utm47[0]) / 2
N_mid_47 = (A_utm47[1] + B_utm47[1]) / 2
E_mid_48 = (A_utm48[0] + B_utm48[0]) / 2
N_mid_48 = (A_utm48[1] + B_utm48[1]) / 2

lon_mid_47, lat_mid_47, dummy = UTM2LLA(E_mid_47, N_mid_47, 0, 47)
lon_mid_48, lat_mid_48, dummy = UTM2LLA(E_mid_48, N_mid_48, 0, 48)

In [26]:
# หา scale ของจุดกึ่งกลาง
scaleX_mid_47, scaleY_mid_47 = utm_scale_factor(lat_mid_47,lon_mid_47,47)
scaleX_mid_48, scaleY_mid_48 = utm_scale_factor(lat_mid_48,lon_mid_48,48)

In [27]:
K_47 = (1/6) * (scaleX_A_47 + (4 * scaleX_mid_47) + scaleY_B_47)
K_48 = (1/6) * (scaleX_A_48 + (4 * scaleX_mid_48) + scaleY_B_48)

#### Distance on Ground (using scale factor) ***

In [28]:
ground_dist_utm47 = grid_dist_utm47 / (ESF * K_47)
ground_dist_utm48 = grid_dist_utm48 / (ESF * K_48)
print(ground_dist_utm47)
print(ground_dist_utm48)

766028.5169568164
766028.346675378


#### Distance on Ellipsoid (using scale factor) ***

In [29]:
ellip_dist_utm47 = ground_dist_utm47 * ESF
ellip_dist_utm48 = ground_dist_utm48 * ESF
print(ellip_dist_utm47)
print(ellip_dist_utm48)

765962.5162615532
765962.3459947861


#### Find Ground Distance A-B (with a reliable method)

In [30]:
# using Charles Karney’s algorithm for ellipsoidal distance

from pygeodesy.ellipsoidalKarney import LatLon

pA = LatLon(A_lla[0], A_lla[1])
pB = LatLon(B_lla[0], B_lla[1])

true_ellip_dist = pA.distanceTo(pB)

true_ground_dist = true_ellip_dist / ESF

print(true_ellip_dist)

765962.6959037109
